In [1]:
# ===============================
# ANN MODEL FOR CUSTOMER CHURN ACS WILDA26_01B Team 5
# ===============================

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
# 1. Load dataset
df = pd.read_csv("Dataset_ATS_v2.csv")

In [4]:
# 2. Display basic information
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape of dataset: (7043, 10)

First 5 rows:
   gender  SeniorCitizen Dependents  tenure PhoneService MultipleLines  \
0  Female              0         No       1           No            No   
1    Male              0         No      41          Yes            No   
2  Female              0        Yes      52          Yes            No   
3  Female              0         No       1          Yes            No   
4    Male              0         No      67          Yes            No   

  InternetService        Contract  MonthlyCharges Churn  
0             DSL  Month-to-month              25   Yes  
1             DSL        One year              25    No  
2             DSL  Month-to-month              19    No  
3             DSL        One year              76   Yes  
4     Fiber optic  Month-to-month              51    No  

Column names:
['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']

Data

In [5]:
# 3. Standardise column names
df.columns = df.columns.str.strip()

In [6]:
# 4. Remove extra spaces from categorical values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [7]:
# 5. Check missing values
print("\nMissing values before handling:")
print(df.isnull().sum())


Missing values before handling:
gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64


In [8]:
# 6. Check duplicate records
print("\nNumber of duplicate rows:", df.duplicated().sum())


Number of duplicate rows: 302


In [9]:
# Remove duplicates
df = df.drop_duplicates()

In [10]:
# 7. Separate features and target
# X contains input features
# y contains the target column that the ANN will predict
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [11]:
# 8. Convert target values to numeric labels
y = y.map({"Yes": 1, "No": 0})

print("\nTarget value count:")
print(y.value_counts())


Target value count:
Churn
0    4950
1    1791
Name: count, dtype: int64


In [12]:
# 9. Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical columns: ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract']


In [13]:
# 10. Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining feature set shape:", X_train.shape)
print("Testing feature set shape:", X_test.shape)
print("Training target set shape:", y_train.shape)
print("Testing target set shape:", y_test.shape)


Training feature set shape: (5392, 9)
Testing feature set shape: (1349, 9)
Training target set shape: (5392,)
Testing target set shape: (1349,)


In [14]:
# 11. Create preprocessing pipelines
# Numeric pipeline:
# - fill missing numeric values with median
# - apply standard scaling
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [15]:
# Categorical pipeline:
# - fill missing categorical values with most frequent value
# - encode categorical variables using one-hot encoding
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [16]:
# 12. Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [17]:
# 13. Apply preprocessing
# Fit on training set, transform both training and testing sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [18]:
# 14. Convert processed data to arrays
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

print("\nProcessed training set shape:", X_train_processed.shape)
print("Processed testing set shape:", X_test_processed.shape)


Processed training set shape: (5392, 10)
Processed testing set shape: (1349, 10)


In [19]:
# 15. Define ANN architecture
ann_model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42,
    early_stopping=True
)

In [20]:
# 16. Train ANN model
ann_model.fit(X_train_processed, y_train)

MLPClassifier(early_stopping=True, hidden_layer_sizes=(64, 32, 16),
              max_iter=300, random_state=42)

In [21]:
# 17. Predict customer churn
y_pred = ann_model.predict(X_test_processed)

In [22]:
# 18. Evaluate model performance
print("\nANN Model Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


ANN Model Accuracy:
0.7857672349888807

Confusion Matrix:
[[893  98]
 [191 167]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       991
           1       0.63      0.47      0.54       358

    accuracy                           0.79      1349
   macro avg       0.73      0.68      0.70      1349
weighted avg       0.77      0.79      0.77      1349



In [23]:
# 19. Save predictions
results_df = X_test.copy()
results_df["Actual_Churn"] = y_test.values
results_df["Predicted_Churn"] = y_pred

results_df.to_csv("ann_churn_predictions.csv", index=False)

print("\nPredictions saved successfully:")
print("- ann_churn_predictions.csv")


Predictions saved successfully:
- ann_churn_predictions.csv
